In [ ]:
PAIR = "ru_ua"
SA, SB = {"ru_ua": ("ru", "ua"), "il_ps": ("ps", "il")}[PAIR]
LA, LB = SA.upper(), SB.upper()
RESULTS_DIR = {"ru_ua": "results", "il_ps": "results_il_ps"}[PAIR]

import os, json
import numpy as np


def load_all_examples_with_prefix(files_with_prefix):
    all_examples = {}
    for prefix, path in files_with_prefix:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                new_id = f"{prefix}_{obj.get('id')}"
                obj["id"] = new_id
                all_examples[new_id] = obj
    return all_examples


def _find_latest_run(sd_path, prefix):
    runs = [d for d in os.listdir(sd_path) if d.startswith(prefix) and os.path.isdir(os.path.join(sd_path, d))]
    if not runs:
        return None
    runs.sort()
    fname = f"{SB}_results.jsonl" if prefix.startswith(SB) else f"{SA}_results.jsonl"
    fpath = os.path.join(sd_path, runs[-1], fname)
    return fpath if os.path.isfile(fpath) else None


def collect_latest_runs_all_subdirs(model_dir):
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f"model_dir not found: {model_dir}")
    sb_files, sa_files = [], []
    for sd in sorted(d for d in os.listdir(model_dir) if os.path.isdir(os.path.join(model_dir, d))):
        sd_path = os.path.join(model_dir, sd)
        sb_path = _find_latest_run(sd_path, f"{SB}_run_")
        sa_path = _find_latest_run(sd_path, f"{SA}_run_")
        if sb_path and sa_path:
            sb_files.append((sd, sb_path))
            sa_files.append((sd, sa_path))
    return sb_files, sa_files


def scores_from_examples(examples, score_key):
    return {ex_id: float(obj[score_key]) for ex_id, obj in examples.items() if obj.get(score_key) is not None}


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    if m == 0:
        return np.array([])
    order = np.argsort(pvals)
    q = pvals[order] * m / np.arange(1, m + 1)
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0.0, 1.0)
    out = np.empty_like(q)
    out[order] = q
    return out


def _repo_root():
    p = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(p, RESULTS_DIR)):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            raise FileNotFoundError(f"no '{RESULTS_DIR}/' dir found above {os.getcwd()}")
        p = parent

In [ ]:
JUDGE_MODEL  = "llama3-8b"
JUDGE_PROMPT = "BASE_JUDGE"
SB_KEY = f"score_{SB}_mt_vs_{SB}_human"
SA_KEY = f"score_{SA}_mt_vs_{SA}_human"

BASELINE_MITIGATION  = "baseline"
INJECTION_MITIGATION = "self_distance"
SDAS_MITIGATION  = "sdas_test"
SDAS_CELLS = {
    "ru_ua": {
        "qwen25-7b": "L17_a1",
        "aya-expanse-8b": "L14_a2",
        "mistral-7b-instruct-v0.3": "L16_a2",
        "gemma-4-e4b-it": "L13_a2",
        "falcon3-7b-instruct": "L16_a2",
    },
    "il_ps": {
        "qwen25-7b": "L16_a1",
        "aya-expanse-8b": "L14_a2",
        "mistral-7b-instruct-v0.3": "L16_a2",
        "gemma-4-e4b-it": "L12_a2",
        "falcon3-7b-instruct": "L16_a2",
    },
}[PAIR]
SDAS_CELLS.update({"gpt-5.6-luna": None, "claude-sonnet-5": None, "kimi-k2.6": None})
SCENARIOS = {
    model: {
        "baseline":  (BASELINE_MITIGATION,  model),
        "injection": (INJECTION_MITIGATION, model),
        "sdas":  (SDAS_MITIGATION,  f"{model}_{cell}" if cell else None),
    }
    for model, cell in SDAS_CELLS.items()
}
SCENARIO_ORDER = ["baseline", "injection", "sdas"]

RESULTS_ROOT = os.path.join(_repo_root(), RESULTS_DIR)


def scenario_dir(mitigation, model_dir_name):
    if model_dir_name is None:
        return None
    return os.path.join(RESULTS_ROOT, mitigation, f"{JUDGE_MODEL}_judge", JUDGE_PROMPT, model_dir_name)


def load_scenario(mitigation, model_dir_name):
    d = scenario_dir(mitigation, model_dir_name)
    if d is None or not os.path.isdir(d):
        return None
    sb_files, sa_files = collect_latest_runs_all_subdirs(d)
    if not sb_files:
        return None
    sb = scores_from_examples(load_all_examples_with_prefix(sb_files), SB_KEY)
    sa = scores_from_examples(load_all_examples_with_prefix(sa_files), SA_KEY)
    if not sb or not sa:
        return None
    runs = {p: (os.path.basename(os.path.dirname(r)), os.path.basename(os.path.dirname(u))) for (p, r), (_, u) in zip(sa_files, sb_files)}
    return {"sa": sa, "sb": sb, "prompts": [p for p, _ in sb_files], "runs": runs, "dir": d}


data = {m: {s: load_scenario(*SCENARIOS[m][s]) for s in SCENARIO_ORDER} for m in SCENARIOS}

print(f"judge : {JUDGE_MODEL}_judge/{JUDGE_PROMPT}")
for m in SCENARIOS:
    for s in SCENARIO_ORDER:
        rec = data[m][s]
        d = scenario_dir(*SCENARIOS[m][s])
        where = "-" if d is None else os.path.relpath(d, RESULTS_ROOT)
        if d is None:
            print(f"  {m:<26}{s:<10} -  (no steer cell chosen yet)")
        elif rec is None:
            print(f"  {m:<26}{s:<10} {where}  -> no latest {SB}+{SA} runs")
        else:
            print(f"  {m:<26}{s:<10} {where}  {SA}={len(rec['sa'])} {SB}={len(rec['sb'])} rows, {len(rec['prompts'])} prompts")

In [ ]:
REFERENCE = "injection"
SCENARIO = "sdas"

def gap_report(rec):
    if rec is None:
        return None
    d_sb, d_sa = float(np.mean(list(rec["sb"].values()))), float(np.mean(list(rec["sa"].values())))
    return {f"d_{SA}": d_sa, f"d_{SB}": d_sb, "gap": d_sb - d_sa, f"n_{SA}": len(rec["sa"]), f"n_{SB}": len(rec["sb"]), "prompts": rec["prompts"]}


gap_cells = {(m, s): gap_report(data[m][s]) for m in SCENARIOS for s in SCENARIO_ORDER}


def _gap_cell_text(rep):
    return "-" if rep is None else f"{rep[('d_' + SA)]:.3f} / {rep[('d_' + SB)]:.3f} / {rep['gap']:+.3f}"


def _dgap(a, b):
    return a["gap"] - b["gap"] if (a and b) else None


_W = 30
print(f"cell    : d({LA}) / d({LB}) / gap,   gap = d({LB}) - d({LA});   dgap sdas-inj = gap(sdas) - gap(injection)\n")
print(f"{'model':<26}" + "".join(s.center(_W) for s in SCENARIO_ORDER) + f"{'dgap inj':>10}{'dgap sdas':>12}{'sdas-inj':>12}")
print("-" * (26 + _W * len(SCENARIO_ORDER) + 34))
for m in SCENARIOS:
    reps = {s: gap_cells[(m, s)] for s in SCENARIO_ORDER}
    line = f"{m:<26}" + "".join(_gap_cell_text(reps[s]).center(_W) for s in SCENARIO_ORDER)
    for v, w in ((_dgap(reps["injection"], reps["baseline"]), 10), (_dgap(reps["sdas"], reps["baseline"]), 12),
                 (_dgap(reps[SCENARIO], reps[REFERENCE]), 12)):
        line += f"{v:>+{w}.3f}" if v is not None else f"{'-':>{w}}"
    print(line)

print(f"\ncoverage: n_{SA} / n_{SB} items, prompts counted, and the run picked per prompt ({SA}, {SB})")
for m in SCENARIOS:
    for s in SCENARIO_ORDER:
        rep, rec = gap_cells[(m, s)], data[m][s]
        if rep is None:
            print(f"  {m:<26}{s:<10} " + ("-  (no steer cell chosen yet)" if SCENARIOS[m][s][1] is None else f"no latest {SB}+{SA} runs"))
            continue
        print(f"  {m:<26}{s:<10} {rep[('n_' + SA)]:>5} / {rep[('n_' + SB)]:<5} {len(rep['prompts'])} prompts   steering cell: {(SCENARIOS[m][s][1] or '-') if s == 'sdas' else '-'}")
        for p, (r, u) in rec["runs"].items():
            print(f"      {p:<22} {r}   {u}")


In [ ]:
from scipy.stats import ttest_rel

N_PERM = 5000
N_BOOT = 2000
SEED = 0
SAVE_CSV_DIR = None


def _event_of(example_key):
    return example_key.split("_ex_", 1)[1] if "_ex_" in example_key else example_key.rsplit("_", 1)[1]


def sign_flip_p(values, n_perm, rng):
    v = np.asarray(values, dtype=float)
    obs = abs(v.mean())
    signs = rng.choice([-1.0, 1.0], size=(n_perm, v.size))
    stats = np.abs(signs @ v) / v.size
    return float((np.sum(stats >= obs - 1e-12) + 1) / (n_perm + 1))


def cluster_sign_flip_p(values, clusters, n_perm, rng):
    v = np.asarray(values, dtype=float)
    labels, inv = np.unique(np.asarray(clusters), return_inverse=True)
    sums = np.bincount(inv, weights=v, minlength=labels.size)
    obs = abs(v.mean())
    signs = rng.choice([-1.0, 1.0], size=(n_perm, labels.size))
    stats = np.abs(signs @ sums) / v.size
    return float((np.sum(stats >= obs - 1e-12) + 1) / (n_perm + 1))


def cluster_bootstrap_ci(values, clusters, n_boot, rng, alpha=0.05):
    v = np.asarray(values, dtype=float)
    labels, inv = np.unique(np.asarray(clusters), return_inverse=True)
    sums = np.bincount(inv, weights=v, minlength=labels.size)
    counts = np.bincount(inv, minlength=labels.size).astype(float)
    idx = rng.integers(0, labels.size, size=(n_boot, labels.size))
    means = sums[idx].sum(axis=1) / counts[idx].sum(axis=1)
    return float(np.percentile(means, 100 * alpha / 2)), float(np.percentile(means, 100 * (1 - alpha / 2)))


def paired_gap_stats(scn, ref, n_perm=N_PERM, n_boot=N_BOOT, seed=SEED):
    keys = sorted(k for k in scn["sa"] if k in scn["sb"] and k in ref["sa"] and k in ref["sb"])
    if not keys:
        return None
    sa_s = np.array([scn["sa"][k] for k in keys]); sb_s = np.array([scn["sb"][k] for k in keys])
    sa_r = np.array([ref["sa"][k] for k in keys]); sb_r = np.array([ref["sb"][k] for k in keys])
    g_s, g_r = sb_s - sa_s, sb_r - sa_r
    delta = g_s - g_r
    dabs = np.abs(g_s) - np.abs(g_r)
    events = [_event_of(k) for k in keys]
    rng = np.random.default_rng(seed)
    n = delta.size
    sd = float(delta.std(ddof=1)) if n > 1 else float("nan")
    p_t = float(ttest_rel(g_s, g_r).pvalue) if n > 1 and np.any(delta != 0) else float("nan")
    return {
        "n_complete": n, "n_events": len(set(events)),
        f"dd_{SA}": float((sa_s - sa_r).mean()), f"dd_{SB}": float((sb_s - sb_r).mean()),
        "gap_ref": float(g_r.mean()), "gap_scn": float(g_s.mean()),
        "d_gap": float(delta.mean()), "sd": sd, "se": sd / np.sqrt(n),
        "p_ex": sign_flip_p(delta, n_perm, rng),
        "p_cluster": cluster_sign_flip_p(delta, events, n_perm, rng),
        "ci_cluster": cluster_bootstrap_ci(delta, events, n_boot, rng),
        "p_ttest": p_t,
        "d_absgap": float(dabs.mean()), "p_absgap_ex": sign_flip_p(dabs, n_perm, rng),
        "share_shrank": float(np.mean(dabs < -1e-12)), "share_grew": float(np.mean(dabs > 1e-12)),
        "examples": {"key": keys, f"{SA}_injection": sa_r, f"{SB}_injection": sb_r, f"{SA}_sdas": sa_s, f"{SB}_sdas": sb_s,
                     "gap_injection": g_r, "gap_sdas": g_s, "delta_gap": delta, "delta_absgap": dabs},
    }


stats = {}
for m in SCENARIOS:
    scn, ref = data[m][SCENARIO], data[m][REFERENCE]
    if scn is not None and ref is not None:
        stats[m] = paired_gap_stats(scn, ref)

family = [(m, r) for m, r in stats.items() if r is not None]
for (m, r), q in zip(family, bh_fdr([r["p_ex"] for _, r in family])):
    r["q_ex_all"] = float(q)

_HDR = (f"{'model':<26}{'n_cc':>6}{'events':>7}{'gap inj':>9}{'gap sdas':>10}{('Δd_' + LA):>8}{('Δd_' + LB):>8}{'Δgap':>8}{'SE':>7}"
        f"{'95%CI(events)':>19}{'p_ex':>8}{'q_BH':>8}{'p_clus':>8}{'p_t':>8}{'Δ|gap|':>8}{'p_|gap|':>8}{'shrank/grew':>13}")
print(f"BH family: {len(family)} test(s) - " + ", ".join(m for m, _ in family))
print(f"delta = (d_{LB} - d_{LA})_sdas - (d_{LB} - d_{LA})_injection per complete-case example;"
      f" Δ|gap| = |g|_sdas - |g|_injection per example\n")
print(_HDR)
for m in SCENARIOS:
    if m not in stats:
        why = "no steer cell chosen yet" if SCENARIOS[m][SCENARIO][1] is None else f"missing latest runs ({SCENARIO} or {REFERENCE})"
        print(f"{m:<26}   -  ({why})"); continue
    r = stats[m]
    if r is None:
        print(f"{m:<26}   no complete cases"); continue
    lo, hi = r["ci_cluster"]
    q = r.get("q_ex_all", float("nan"))
    ci_txt = f"[{lo:+.3f},{hi:+.3f}]"
    share_txt = f"{r['share_shrank']:.3f}/{r['share_grew']:.3f}"
    print(f"{m:<26}{r['n_complete']:>6}{r['n_events']:>7}{r['gap_ref']:>+9.3f}{r['gap_scn']:>+10.3f}{r[('dd_' + SA)]:>+8.4f}{r[('dd_' + SB)]:>+8.4f}{r['d_gap']:>+8.4f}{r['se']:>7.4f}"
          f"{ci_txt:>19}{r['p_ex']:>8.4f}{q:>8.4f}{r['p_cluster']:>8.4f}{r['p_ttest']:>8.4f}{r['d_absgap']:>+8.4f}{r['p_absgap_ex']:>8.4f}{share_txt:>13}")

if SAVE_CSV_DIR:
    import csv
    os.makedirs(SAVE_CSV_DIR, exist_ok=True)
    for m, r in stats.items():
        if r is None:
            continue
        path = os.path.join(SAVE_CSV_DIR, f"examples_{m}_{SCENARIO}_vs_{REFERENCE}.csv")
        ex = r["examples"]
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["example", "event"] + [c for c in ex if c != "key"])
            for i, k in enumerate(ex["key"]):
                w.writerow([k, _event_of(k)] + [f"{ex[c][i]:.4f}" for c in ex if c != "key"])
        print(f"wrote {path}")


In [ ]:
def _event_sums(scores, col):
    sums, counts = np.zeros(len(col)), np.zeros(len(col))
    for k, v in scores.items():
        j = col[_event_of(k)]
        sums[j] += v
        counts[j] += 1
    return sums, counts


def _abs_gap(sb_sum, sb_cnt, sa_sum, sa_cnt):
    return np.abs(sb_sum / sb_cnt - sa_sum / sa_cnt)


def abs_gap_stats(scn, ref, n_perm=N_PERM, n_boot=N_BOOT, seed=SEED):
    events = sorted({_event_of(k) for rec in (scn, ref) for side in ("sa", "sb") for k in rec[side]})
    col = {e: j for j, e in enumerate(events)}
    s_sb, s_sa, r_sb, r_sa = (_event_sums(rec[side], col) for rec, side in ((scn, "sb"), (scn, "sa"), (ref, "sb"), (ref, "sa")))
    g_scn = _abs_gap(s_sb[0].sum(), s_sb[1].sum(), s_sa[0].sum(), s_sa[1].sum())
    g_ref = _abs_gap(r_sb[0].sum(), r_sb[1].sum(), r_sa[0].sum(), r_sa[1].sum())
    observed = float(g_scn - g_ref)
    rng = np.random.default_rng(seed)

    idx = rng.integers(0, len(events), size=(n_boot, len(events)))
    tot = [arr[idx].sum(axis=1) for pair in (s_sb, s_sa, r_sb, r_sa) for arr in pair]
    boots = _abs_gap(*tot[0:4]) - _abs_gap(*tot[4:8])
    lo, hi = np.percentile(boots, [2.5, 97.5])

    swap = rng.integers(0, 2, size=(n_perm, len(events))).astype(float)
    perm_scn, perm_ref = [], []
    for (sv, rv) in ((s_sb[0], r_sb[0]), (s_sb[1], r_sb[1]), (s_sa[0], r_sa[0]), (s_sa[1], r_sa[1])):
        moved = swap @ (rv - sv)
        perm_scn.append(sv.sum() + moved)
        perm_ref.append(rv.sum() - moved)
    perms = _abs_gap(*perm_scn) - _abs_gap(*perm_ref)
    p = float((np.sum(np.abs(perms) >= abs(observed) - 1e-12) + 1) / (n_perm + 1))
    return {"n_events": len(events), "absgap_ref": float(g_ref), "absgap_scn": float(g_scn),
            "d_absgap": observed, "ci": (float(lo), float(hi)), "p": p}


paper_stats = {}
for m in SCENARIOS:
    scn, ref = data[m][SCENARIO], data[m][REFERENCE]
    if scn is not None and ref is not None:
        paper_stats[m] = abs_gap_stats(scn, ref)

paper_family = list(paper_stats)
for m, q in zip(paper_family, bh_fdr([paper_stats[m]["p"] for m in paper_family])):
    paper_stats[m]["q"] = float(q)

print(f"Δ|g| = |d_{LB} - d_{LA}|_sdas - |d_{LB} - d_{LA}|_injection;  B = {N_BOOT} event resamples, P = {N_PERM} event sign-flips, "
      f"BH family: {len(paper_family)} test(s) (one per model)\n")
print(f"{'model':<26}{'events':>7}{'|g| base':>10}{'|g| inj':>10}{'|g| sdas':>10}{'Δ|g|':>9}{'95% CI (events)':>20}{'p':>8}{'q_BH':>8}")
for m, r in paper_stats.items():
    lo, hi = r["ci"]
    star = "*" if r["q"] < 0.05 else ""
    base = gap_cells[(m, "baseline")]
    base_txt = f"{abs(base['gap']):.4f}" if base else "-"
    print(f"{m:<26}{r['n_events']:>7}{base_txt:>10}{r['absgap_ref']:>10.4f}{r['absgap_scn']:>10.4f}{r['d_absgap']:>+9.4f}"
          f"{f'[{lo:+.4f},{hi:+.4f}]':>20}{r['p']:>8.4f}{r['q']:>8.4f} {star}")


In [ ]:
UNIT = "event"

def unit_table(rec, unit=None):
    """{unit: (d_B, d_A)} - event: the side means over all of the event's rows; example: the row pair's own two scores."""
    unit = unit or UNIT
    if unit == "example":
        return {k: (rec["sb"][k], rec["sa"][k]) for k in rec["sb"] if k in rec["sa"]}
    acc = {}
    for j, side in enumerate(("sb", "sa")):
        for k, v in rec[side].items():
            s = acc.setdefault(_event_of(k), [[0.0, 0], [0.0, 0]])
            s[j][0] += v
            s[j][1] += 1
    return {e: (s[0][0] / s[0][1], s[1][0] / s[1][1]) for e, s in acc.items() if s[0][1] and s[1][1]}


def signed_gaps(tab, keys=None):
    keys = list(tab) if keys is None else keys
    return np.array([tab[k][0] - tab[k][1] for k in keys], dtype=float)


def paired_divergence_stats(scn, ref, n_perm=N_PERM, n_boot=N_BOOT, seed=SEED):
    ts, tr = unit_table(scn), unit_table(ref)
    keys = sorted(set(ts) & set(tr))
    if not keys:
        return None
    g_s, g_r = signed_gaps(ts, keys), signed_gaps(tr, keys)
    abs_s, abs_r = np.abs(g_s), np.abs(g_r)
    delta = abs_s - abs_r
    events = [_event_of(k) for k in keys] if UNIT == "example" else keys
    rng = np.random.default_rng(seed)
    n = delta.size
    sd = float(delta.std(ddof=1)) if n > 1 else float("nan")
    return {
        "n_units": int(n), "n_events": len(set(events)),
        "D_ref": float(abs_r.mean()), "D_scn": float(abs_s.mean()), "dD": float(delta.mean()),
        "sd": sd, "se": sd / np.sqrt(n),
        "p": cluster_sign_flip_p(delta, events, n_perm, rng),
        "ci": cluster_bootstrap_ci(delta, events, n_boot, rng),
        "p_ttest": float(ttest_rel(abs_s, abs_r).pvalue) if n > 1 and np.any(delta != 0) else float("nan"),
        "share_shrank": float(np.mean(delta < -1e-12)), "share_grew": float(np.mean(delta > 1e-12)),
    }


def divergence_D(rec):
    return float(np.abs(signed_gaps(unit_table(rec))).mean()) if rec is not None else None


div_stats = {}
for m in SCENARIOS:
    scn, ref = data[m][SCENARIO], data[m][REFERENCE]
    if scn is not None and ref is not None:
        div_stats[m] = paired_divergence_stats(scn, ref)

div_family = [m for m, r in div_stats.items() if r is not None]
for m, q in zip(div_family, bh_fdr([div_stats[m]["p"] for m in div_family])):
    div_stats[m]["q"] = float(q)

print(f"unit = {UNIT};  ΔD = mean_u |d_{LB},u - d_{LA},u|_sdas - mean_u |d_{LB},u - d_{LA},u|_injection;  B = {N_BOOT} event resamples, "
      f"P = {N_PERM} event sign-flips, BH family: {len(div_family)} test(s) (one per model)\n")
print(f"{'model':<26}{'units':>6}{'events':>7}{'D base':>9}{'D inj':>9}{'D sdas':>9}{'ΔD':>9}{'SE':>8}"
      f"{'95% CI (events)':>20}{'p':>8}{'q_BH':>8}{'p_t':>8}{'shrank/grew':>13}")
for m, r in div_stats.items():
    if r is None:
        print(f"{m:<26}   no paired units"); continue
    lo, hi = r["ci"]
    star = "*" if r["q"] < 0.05 else ""
    d_base = divergence_D(data[m]["baseline"])
    base_txt = f"{d_base:.4f}" if d_base is not None else "-"
    share_txt = f"{r['share_shrank']:.3f}/{r['share_grew']:.3f}"
    print(f"{m:<26}{r['n_units']:>6}{r['n_events']:>7}{base_txt:>9}{r['D_ref']:>9.4f}{r['D_scn']:>9.4f}{r['dD']:>+9.4f}{r['se']:>8.4f}"
          f"{f'[{lo:+.4f},{hi:+.4f}]':>20}{r['p']:>8.4f}{r['q']:>8.4f}{r['p_ttest']:>8.4f}{share_txt:>13} {star}")
